Şimdi burada modelin elde edilen featurelara göre eğitip daha sonrasında test edeceğim. Test için accuracy score, f1 score, roc accuracy score, classification report ve confusion matrislerini gözlemleyeceğim. Model olarak daha önceden de belirtmiş olduğum XGBoost modelini kullanacağım. Daha önceden drive kayıt ettiğim X_train.npy`, `y_train.npy`, `X_test.npy`, `y_test.npy`, `feature_isimleri.json` verileri giriş olarak alacağım.

Eğitim yaparken 99 featuredan belli bir eşiğin altında kalan featureları eleyeceğim, ardından kalan featurelar ile eğitime devam edeceğim. Bu elemeyi yaptığım ön çalışmalarda fark ettiğim üzere elemeden sonra elemeden öncesiyle neredeyse aynı performansı elde ettim. Bundan dolayı eleme yapmaktayım literatürde de bu tekniğin kullanıldığını gördüm. 


In [ ]:
# Gerekli kütüphanelerin kurulumu ve yüklenmesi
!pip install -q xgboost scikit-learn

import numpy as np
import json
import xgboost as xgb
from google.colab import drive

# aşağıda kullınlan metrikler yer almaktadır.
from sklearn.metrics import (
    accuracy_score,        # doğru tahmin oranı
    f1_score,              # precision ve recall dengesi
    roc_auc_score,         # ayırt edicilik gücü
    classification_report, # sınıf bazlı detaylı rapor
    confusion_matrix       # TN, FP, FN, TP tablosu
)

drive.mount('/content/drive')

In [ ]:
# feature_extraction.ipynb'de oluşturulan ve Drive'a kaydedilen matrisler yüklenmekte.
# X: her satır bir cevabı, her sütun bir feature'ı temsil eder
# y: 1=doğru cevap, 0=yanlış cevap
X_train = np.load('/content/drive/MyDrive/proje5/data/X_train.npy')
y_train = np.load('/content/drive/MyDrive/proje5/data/y_train.npy')
X_test  = np.load('/content/drive/MyDrive/proje5/data/X_test.npy')
y_test  = np.load('/content/drive/MyDrive/proje5/data/y_test.npy')

with open('/content/drive/MyDrive/proje5/data/feature_isimleri.json') as f:
    feature_isimleri = json.load(f)

print(f"X_train : {X_train.shape}  ({X_train.shape[0]} örnek, {X_train.shape[1]} feature)")
print(f"y_train : {y_train.shape}  → Doğru: {y_train.sum()}, Yanlış: {(y_train==0).sum()}")
print(f"X_test  : {X_test.shape}")
print(f"y_test  : {y_test.shape}  → Doğru: {y_test.sum()}, Yanlış: {(y_test==0).sum()}")
print(f"Feature : {len(feature_isimleri)} adet")

Modelin doğru ve yanlış dağılımında bir dengesizlik yer alıyor. Eğitim setinde 957 doğru 1543 yanlış yer alırken, test setinde 961 doğru 1539 yanlış yer alıyor. Bundan dolayı bir ön işlem uyguluyorum doğru ve yanlışlar için, bunu metodu ai ile konuşmalarım sırasında belirledim. Model kolay yolu tercih etmemesi için doğru olanları yanlış tahmin edince loss değeri yanlış olanları doğru tahmin edincedeki loss değerinden daha fazla değişmektedir.

In [ ]:
# scale_pos_weight = yanlış sayısı / doğru sayısı
# XGBoost'a "doğru cevapları bu oranda daha önemli say" bilgisi verilmektedir.
yanlis_sayisi = (y_train == 0).sum()
dogru_sayisi  = (y_train == 1).sum()

scale_pos_weight = yanlis_sayisi / dogru_sayisi

print(f"Doğru  (1): {dogru_sayisi:5d}")
print(f"Yanlış (0): {yanlis_sayisi:5d}")
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

Ablasyon çalışmasında 99 feature'dan eşik=0.011 ile seçilen 10 feature'ın,
tüm feature seti ile neredeyse aynı performansı (AUC: 0.877 vs 0.876) sağladığı görülmüştür. Seçilen 10 model aşağıdaki kod blopunda gözükmektedir.

In [ ]:
# Ablasyon sonucu seçilen 10 feature, bunlar en yüksek sahip tokenlardı.
secilen_featureler = [
    'num_tokens',           # cevap uzunluğu — yanlış cevaplar daha uzun
    't12_son_donum_pos',    # top1-top2 farkının son dönüm noktası pozisyonu
    't1_son_donum_pos',     # top1 olasılığının son dönüm noktası pozisyonu
    'ent_final',            # son token entropy — makale metriği
    't10_son_donum_pos',    # top10 toplamının son dönüm noktası pozisyonu
    'ent_cumulative_drop',  # entropy kümülatif düşüşü — A* heuristic
    'ent_son_mean',         # son %33'te entropy ortalaması
    'ent_dusuk_oran',       # düşük entropy olan token oranı
    't12_cumulative_drop',  # top1-top2 farkı kümülatif düşüşü
    't1_cumulative_drop',   # top1 olasılığı kümülatif düşüşü
]

# Feature isimlerinden indeks bulunarak matrisin sadece ilgili sütunları seçilmekte
mask_10    = np.array([f in secilen_featureler for f in feature_isimleri])
X_train_10 = X_train[:, mask_10]
X_test_10  = X_test[:, mask_10]
isimler_10 = [feature_isimleri[i] for i in range(len(feature_isimleri)) if mask_10[i]]

print(f"Seçilen feature sayısı : {mask_10.sum()}")
print(f"X_train_10 boyutu      : {X_train_10.shape}")
print(f"X_test_10 boyutu       : {X_test_10.shape}")

# ── Model tanımla ─────────────────────────────────────────
# Early stopping: 30 ağaç boyunca test seti iyileşmezse eğitim durdurulur (overfitting önleme).
model_final = xgb.XGBClassifier(
    n_estimators=1000,       # üst sınır — early stopping daha erken durdurur
    max_depth=4,             # ağaç derinliği — küçük tutarak overfitting önlenir
    learning_rate=0.05,      # her ağacın katkısı — küçük = daha stabil
    subsample=0.8,           # her ağaçta verinin %80'ini kullan
    colsample_bytree=0.8,    # her ağaçta feature'ların %80'ini kullan
    scale_pos_weight=scale_pos_weight,  # sınıf dengesizliği düzeltme
    eval_metric='logloss',
    early_stopping_rounds=30,
    random_state=42,         # tekrarlanabilirlik
    n_jobs=-1                # tüm CPU çekirdekleri
)

# ── Eğit ─────────────────────────────────────────────────
# eval_set ile hem eğitim hem test kaybı takip edilmekte; early stopping buna göre çalışır.
print("\nModel eğitiliyor (10 feature, early stopping aktif)...")
model_final.fit(
    X_train_10, y_train,
    eval_set=[(X_train_10, y_train), (X_test_10, y_test)],
    verbose=20
)

print(f"En iyi ağaç sayısı  : {model_final.best_iteration}")
print(f"En iyi test logloss : {model_final.best_score:.5f}")

Test seti ilk başta belirttiğim metriklere göre aşağıda değerlendireceğim.

In [ ]:
y_pred       = model_final.predict(X_test_10)
y_pred_proba = model_final.predict_proba(X_test_10)[:, 1]  # sadece "doğru" sınıfının olasılığı

acc     = accuracy_score(y_test, y_pred)
f1      = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"{'='*40}")
print(f"SONUÇLAR (10 feature)")
print(f"{'='*40}")
print(f"Accuracy  : {acc:.4f}  (%{acc*100:.1f})")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"En iyi ağaç: {model_final.best_iteration}")

print(f"\nDetaylı Sınıf Raporu:")
print(classification_report(y_test, y_pred,
      target_names=['Yanlış (0)', 'Doğru (1)']))

cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:")
print(f"  TN={cm[0,0]:4d}  FP={cm[0,1]:4d}   (Gerçek Yanlış)")
print(f"  FN={cm[1,0]:4d}  TP={cm[1,1]:4d}   (Gerçek Doğru)")

In [ ]:
# Eğitilen model ve kullanılan feature listesi Drive'a kaydedilmekte.
model_final.save_model('/content/drive/MyDrive/proje5/data/xgboost_final.json')

with open('/content/drive/MyDrive/proje5/data/secilen_featureler.json', 'w') as f:
    json.dump(isimler_10, f, indent=2)

print("Model kaydedildi → xgboost_final.json")
print("Feature listesi kaydedildi → secilen_featureler.json")